# 03 · RAG evaluation: retrieval metrics and faithfulness

After this notebook you can build a small golden set, score a retriever with hit rate, recall, precision, MRR and nDCG (all from scratch, checked against scikit-learn), and compare BM25, dense and hybrid retrieval on recall@k curves. You can also score generated answers for faithfulness, answer relevancy, context precision and context recall, the RAGAS-style metrics, implemented simply enough to explain on a whiteboard.

**Time:** ~35 min · **Runs:** offline on CPU in < 1 min (uses the cached MiniLM encoder) · **Needs:** [RAG pipeline from scratch](../09_rag/06_rag_pipeline_from_scratch.ipynb), [01 · eval metrics](01_llm_eval_metrics_em_f1_bleu_rouge.ipynb)

## Why this matters in interviews

- "How do you evaluate a RAG system?" is the most common eval question of all (`rg11`). The expected shape of the answer is **grade the retriever and the generator separately**.
- recall@k / MRR / nDCG definitions and when to use each (`ve29`) are asked directly, and often coded.
- Faithfulness versus correctness (`ev30`) and the RAGAS metrics (`rg31`, `ev07`) are the standard depth probes.
- "How would you build the eval set?" (`rg12`, `ev15`) separates people who have shipped RAG from people who have demoed it.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `rg11` | How do you evaluate a RAG system? What do you measure? |
| `rg12` | How do you build a RAG eval set from nothing? |
| `ve29` | What is recall@k, MRR and nDCG — and when do you use each? |
| `rg30` | What is context precision and why does it matter as much as recall? |
| `rg31` | What is RAGAS and what does it actually compute? |
| `ev07` | What is RAGAS and when does each metric help? |
| `ev14` | How do you evaluate hallucination specifically? |
| `ev30` | What is the difference between accuracy and faithfulness? |
| `ev15` | What is a golden dataset and how do you keep it useful? |
| `tl06` | What is the RAG Triad? |

In [ ]:
import sys
from pathlib import Path
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "llm_setup.py").exists():
        sys.path.insert(0, str(_p)); break
from llm_setup import client, MODEL, EMBED_MODEL, PROVIDER

In [ ]:
import functools
import hashlib
import json
import math
import re
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pydantic import BaseModel
from rank_bm25 import BM25Okapi
from sklearn.metrics import ndcg_score

pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 70)
rng = np.random.default_rng(0)

## The picture: two halves, graded separately

```
 question ──► RETRIEVER ──► top-k chunks ──► GENERATOR ──► answer
               │                               │
   retrieval metrics (no LLM needed):   generation metrics:
   hit rate@k, recall@k, precision@k,   faithfulness (claims supported by the context?)
   MRR, nDCG@k, context precision,      answer relevancy (does it address the question?)
   context recall                       correctness vs a reference (if you have one)
```

**In plain English:** grade the librarian and the writer separately (`rg11`). If the right page was never fetched, good writing cannot save the answer. If it was fetched and the answer ignores it, that is a different bug with a different fix. The RAG Triad (`tl06`) is the same split with three names: context relevance, groundedness, answer relevance.

## 1. A tiny knowledge base and a golden set

Twenty-seven help-centre snippets for a fictional SaaS company, with deliberate near-duplicates: five error codes that differ by a digit, three kinds of leave, two ways to get a file back. The golden set has 13 answerable questions, each labelled with the doc ids that answer it, and **graded** relevance: 2 = answers it, 1 = related or partly needed. It also has one out-of-scope question where the right behaviour is "I don't know". The questions are deliberately mixed: paraphrases (no shared keywords), exact identifiers (error codes), and a multi-hop question.

In [ ]:
KB = {
    "d01": "Refunds are issued to the original payment method within 5 business days after we receive the returned item.",
    "d02": "Items can be returned within 30 days of delivery if they are unused and in the original packaging.",
    "d03": "If an item arrives damaged, send us a photo within 48 hours and we will ship a replacement at no cost.",
    "d04": "Standard shipping takes 3 to 7 business days and is free on orders over $50.",
    "d05": "Express shipping delivers in 1 to 2 business days and costs $15.",
    "d06": "We ship to 40 countries. Customs duties and import taxes are paid by the customer on delivery.",
    "d07": "To reset your password, click 'Forgot password' on the login page. The reset link expires after 30 minutes.",
    "d08": "Two-factor authentication can be turned on under Settings > Security with an authenticator app or SMS codes.",
    "d09": "After 5 failed sign-in attempts the account is locked for 15 minutes.",
    "d10": "Error E-4012 means the payment card was declined by the bank.",
    "d11": "Error E-4021 means the payment card has expired. Ask the customer to update the expiry date.",
    "d12": "Error E-4102 means the billing address does not match the address on the card.",
    "d13": "Error E-5003 means the upload exceeded the 2 GB file size limit.",
    "d14": "Error E-5030 means the upload was interrupted by a network timeout. Retry the upload.",
    "d15": "You can cancel your subscription at any time from Billing > Plan. Access continues until the end of the current billing period, and there are no partial refunds.",
    "d16": "The Pro plan costs $12 per user per month when billed annually, or $15 per user per month when billed monthly.",
    "d17": "The Enterprise plan is priced per contract and includes single sign-on with SAML and a dedicated account manager.",
    "d18": "Deleted files stay in the trash for 30 days. After that they are permanently deleted and cannot be restored.",
    "d19": "Previous versions of a file are kept for 90 days and can be restored from the file's History tab.",
    "d20": "Employees receive 25 days of paid vacation per year. Up to 5 unused days roll over to the next year.",
    "d21": "Employees may take up to 10 paid sick days per year. A doctor's note is required after 3 consecutive sick days.",
    "d22": "Parents receive 16 weeks of paid parental leave.",
    "d23": "The public API allows 100 requests per minute per API key. Requests above the limit receive HTTP status 429.",
    "d24": "API keys are created under Settings > Developer and can be revoked at any time.",
    "d25": "Admins can export all workspace data as a ZIP archive from Settings > Data > Export.",
    "d26": "Live chat support is open Monday to Friday, 9 am to 6 pm Central European Time.",
    "d27": "Each Pro user gets 1 TB of storage; the free plan includes 5 GB.",
}
GOLDEN = [  # id, question, graded relevant docs, reference answer, tags
    {"id": "q01", "q": "How long until I get my money back after sending something back?", "rel": {"d01": 2},
     "ref": "The refund goes to the original payment method within 5 business days after the returned item arrives.", "tags": ["paraphrase"]},
    {"id": "q02", "q": "What does error E-4021 mean?", "rel": {"d11": 2},
     "ref": "E-4021 means the payment card has expired.", "tags": ["identifier"]},
    {"id": "q03", "q": "How fast is express delivery and what does it cost?", "rel": {"d05": 2},
     "ref": "Express shipping delivers in 1 to 2 business days and costs $15.", "tags": ["keyword"]},
    {"id": "q04", "q": "I forgot my login credentials. How do I get back into my account?", "rel": {"d07": 2, "d09": 1},
     "ref": "Click 'Forgot password' on the login page; the reset link expires after 30 minutes.", "tags": ["paraphrase"]},
    {"id": "q05", "q": "If I stop paying for my plan, do I lose access straight away?", "rel": {"d15": 2},
     "ref": "No. Access continues until the end of the current billing period.", "tags": ["paraphrase"]},
    {"id": "q06", "q": "What is the Pro price per user with monthly billing?", "rel": {"d16": 2},
     "ref": "The Pro plan costs $15 per user per month when billed monthly.", "tags": ["keyword"]},
    {"id": "q07", "q": "Why was my 3 GB video rejected when I tried to add it?", "rel": {"d13": 2},
     "ref": "The upload exceeded the 2 GB file size limit.", "tags": ["paraphrase"]},
    {"id": "q08", "q": "How many holiday days do staff get each year?", "rel": {"d20": 2},
     "ref": "Employees receive 25 days of paid vacation per year.", "tags": ["paraphrase"]},
    {"id": "q09", "q": "What happens if we go over the API rate limit?", "rel": {"d23": 2},
     "ref": "Requests above 100 per minute per API key receive HTTP status 429.", "tags": ["keyword"]},
    {"id": "q10", "q": "Can I get back an older copy of a document I overwrote last month?", "rel": {"d19": 2, "d18": 1},
     "ref": "Yes. Previous versions are kept for 90 days and can be restored from the History tab.", "tags": ["paraphrase"]},
    {"id": "q11", "q": "Do I need a doctor's note for sick leave?", "rel": {"d21": 2},
     "ref": "A doctor's note is required after 3 consecutive sick days.", "tags": ["keyword"]},
    {"id": "q12", "q": "Is SAML single sign-on included in the Pro plan?", "rel": {"d17": 2, "d16": 1},
     "ref": "No. Single sign-on with SAML is included in the Enterprise plan.", "tags": ["multi-hop"]},
    {"id": "q13", "q": "What does error E-5030 mean?", "rel": {"d14": 2},
     "ref": "E-5030 means the upload was interrupted by a network timeout.", "tags": ["identifier"]},
    {"id": "q14", "q": "What is the CEO's favourite colour?", "rel": {},
     "ref": "I don't know.", "tags": ["out-of-scope"]},
]
ANSWERABLE = [g for g in GOLDEN if g["rel"]]
print(f"{len(KB)} docs, {len(GOLDEN)} golden questions ({len(ANSWERABLE)} answerable)")
print("tags:", dict(Counter(t for g in GOLDEN for t in g["tags"])))

## 2. Retrieval metrics from scratch

For one query with a ranked list of doc ids and a set of relevant ids:

| Metric | Definition | Question it answers |
|---|---|---|
| **hit rate@k** | 1 if *any* relevant doc is in the top k | "did the generator see at least one right chunk?" |
| **recall@k** | relevant docs in the top k / all relevant docs | "did it see *everything* it needs?" (multi-hop) |
| **precision@k** | relevant docs in the top k / k | "how much noise came along?" |
| **MRR** | mean of 1 / rank of the first relevant doc | "how high was the first right answer?" |
| **nDCG@k** | Σ gain_i / log2(i+1), divided by the same sum for the ideal ordering | "how good is the whole ordering, with graded relevance?" |

**Gotcha:** many RAG write-ups (including the theory bank's `ve29` answer) say "recall@k" when they mean *hit rate*: at least one relevant doc per query. With exactly one relevant doc per query the two are identical. With several, they differ, so say which one you report.

Worked example, by hand. The ranking is [d3, d1, d7, d2] and the relevant docs are {d1: 2, d2: 1}.
- hit@1 = 0, hit@2 = 1. recall@2 = 1/2, recall@4 = 1. precision@2 = 1/2. Reciprocal rank = 1/2.
- DCG@4 = 2/log2(3) + 1/log2(5) = 1.2619 + 0.4307 = 1.6925. The ideal order puts d1 then d2: IDCG = 2/log2(2) + 1/log2(3) = 2.6309. So nDCG@4 = **0.6433**.

In [ ]:
def hit_rate_at_k(ranked, rel, k):
    return float(any(d in rel for d in ranked[:k]))

def recall_at_k(ranked, rel, k):
    return len(set(ranked[:k]) & set(rel)) / len(rel)

def precision_at_k(ranked, rel, k):
    return len(set(ranked[:k]) & set(rel)) / k

def reciprocal_rank(ranked, rel):
    return next((1 / (i + 1) for i, d in enumerate(ranked) if d in rel), 0.0)

def ndcg_at_k(ranked, grades: dict, k):
    """Linear gains (grade / log2(position + 1)), the convention sklearn.metrics.ndcg_score uses."""
    dcg = sum(grades.get(d, 0) / math.log2(i + 2) for i, d in enumerate(ranked[:k]))
    ideal = sorted(grades.values(), reverse=True)[:k]
    idcg = sum(g / math.log2(i + 2) for i, g in enumerate(ideal))
    return dcg / idcg if idcg else 0.0

ranked, grades = ["d3", "d1", "d7", "d2"], {"d1": 2, "d2": 1}
assert (hit_rate_at_k(ranked, grades, 1), hit_rate_at_k(ranked, grades, 2)) == (0.0, 1.0)
assert (recall_at_k(ranked, grades, 2), recall_at_k(ranked, grades, 4), precision_at_k(ranked, grades, 2)) == (0.5, 1.0, 0.5)
assert reciprocal_rank(ranked, grades) == 0.5
expected = (2 / math.log2(3) + 1 / math.log2(5)) / (2 + 1 / math.log2(3))
assert abs(ndcg_at_k(ranked, grades, 4) - expected) < 1e-12
# scikit-learn wants per-document relevances and scores: turn the ranking into descending scores
docs = ["d1", "d2", "d3", "d7"]
y_true = [[grades.get(d, 0) for d in docs]]
y_score = [[len(ranked) - ranked.index(d) for d in docs]]
assert abs(ndcg_score(y_true, y_score, k=4) - expected) < 1e-12
print(f"nDCG@4 = {ndcg_at_k(ranked, grades, 4):.4f} (hand: {expected:.4f}, sklearn: {ndcg_score(y_true, y_score, k=4):.4f})")

## 3. Three retrievers: BM25, dense (MiniLM) and hybrid (RRF)

- **BM25** (`rank_bm25`) scores keyword overlap, weighted by rarity. It is excellent on identifiers and exact terms, and blind to paraphrase.
- **Dense:** cosine similarity of MiniLM sentence embeddings. It handles paraphrase and is weak on codes and numbers. (MiniLM is loaded with plain `transformers` and mean-pooled, which gives the same vectors as `SentenceTransformer(...).encode(..., normalize_embeddings=True)`.)
- **Hybrid:** Reciprocal Rank Fusion, score(d) = Σ over retrievers of 1 / (60 + rank). It needs no score normalisation, which is why it is the common default.

In [ ]:
import torch

def load_encoder(name="sentence-transformers/all-MiniLM-L6-v2"):
    """MiniLM from the local Hugging Face cache (downloads ~90 MB once if missing)."""
    from transformers import AutoModel, AutoTokenizer
    from transformers.utils import logging as hf_logging
    hf_logging.disable_progress_bar()
    try:
        return AutoTokenizer.from_pretrained(name, local_files_only=True), AutoModel.from_pretrained(name, local_files_only=True).eval()
    except Exception:
        return AutoTokenizer.from_pretrained(name), AutoModel.from_pretrained(name).eval()

enc_tok, encoder = load_encoder()

def embed(texts) -> np.ndarray:
    """Mean-pooled, L2-normalised MiniLM sentence embeddings (batch)."""
    batch = enc_tok(list(texts), padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad():
        h = encoder(**batch).last_hidden_state
    m = batch["attention_mask"].unsqueeze(-1).float()
    return torch.nn.functional.normalize((h * m).sum(1) / m.sum(1), dim=1).numpy()

_EMB = {}
def embed_cached(texts) -> np.ndarray:
    """Embed with a cache, batching only the texts not seen before (sentences repeat a lot in eval loops)."""
    missing = [t for t in dict.fromkeys(texts) if t not in _EMB]
    if missing:
        _EMB.update(zip(missing, embed(missing)))
    return np.stack([_EMB[t] for t in texts])

def embed_one(text: str) -> np.ndarray:
    return embed_cached([text])[0]

STOP = set("a an the is are was were be to of in on for and or if do does did i my me we our you your it its how what why when can "
           "with at by from this that there than after".split())
def bm25_tokens(text):
    return [w for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in STOP]

DOC_IDS = list(KB)
bm25 = BM25Okapi([bm25_tokens(KB[d]) for d in DOC_IDS])
DOC_EMB = embed_cached([KB[d] for d in DOC_IDS])

@functools.lru_cache(maxsize=None)      # rankings are recomputed many times in the eval loops below
def bm25_rank(q):
    s = bm25.get_scores(bm25_tokens(q))
    return tuple(DOC_IDS[i] for i in np.argsort(-s, kind="stable"))

@functools.lru_cache(maxsize=None)
def dense_rank(q):
    s = DOC_EMB @ embed_one(q)
    return tuple(DOC_IDS[i] for i in np.argsort(-s, kind="stable"))

@functools.lru_cache(maxsize=None)
def hybrid_rank(q, k_rrf=60, w_bm25=1.0, w_dense=1.0):
    fused = Counter()
    for ranking, w in ((bm25_rank(q), w_bm25), (dense_rank(q), w_dense)):
        for r, d in enumerate(ranking, start=1):
            fused[d] += w / (k_rrf + r)
    return tuple(d for d, _ in fused.most_common())

RETRIEVERS = {"BM25": bm25_rank, "dense": dense_rank, "hybrid": hybrid_rank}
for name, f in RETRIEVERS.items():
    print(f"{name:7} top-3 for 'What does error E-4012 mean?': {f('What does error E-4012 mean?')[:3]}")

In [ ]:
def evaluate(retriever, golden, ks=(1, 3, 5)):
    rows = []
    for g in golden:
        ranked = retriever(g["q"])
        row = {"id": g["id"], "tags": ",".join(g["tags"]), "RR": reciprocal_rank(ranked, g["rel"])}
        for k in ks:
            row[f"hit@{k}"] = hit_rate_at_k(ranked, g["rel"], k)
            row[f"recall@{k}"] = recall_at_k(ranked, g["rel"], k)
        row["nDCG@5"] = ndcg_at_k(ranked, g["rel"], 5)
        # cross-check nDCG against scikit-learn for every query and retriever
        y_true = [[g["rel"].get(d, 0) for d in DOC_IDS]]
        y_score = [[len(ranked) - ranked.index(d) for d in DOC_IDS]]
        assert abs(ndcg_score(y_true, y_score, k=5) - row["nDCG@5"]) < 1e-9
        rows.append(row)
    return pd.DataFrame(rows).set_index("id")

per_query = {name: evaluate(f, ANSWERABLE) for name, f in RETRIEVERS.items()}
summary = pd.DataFrame({name: df.drop(columns="tags").mean() for name, df in per_query.items()}).T.rename(columns={"RR": "MRR"})
print(summary[["hit@1", "hit@3", "recall@3", "recall@5", "MRR", "nDCG@5"]].round(3).to_string())

In [ ]:
ks = range(1, 9)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for name, f in RETRIEVERS.items():
    rankings = [f(g["q"]) for g in ANSWERABLE]
    axes[0].plot(ks, [np.mean([recall_at_k(r, g["rel"], k) for r, g in zip(rankings, ANSWERABLE)]) for k in ks], "o-", label=name)
axes[0].set_xlabel("k (chunks passed to the generator)"); axes[0].set_ylabel("mean recall@k")
axes[0].set_title("recall@k curves"); axes[0].legend(); axes[0].set_ylim(0, 1.05)

rr = pd.DataFrame({name: df["RR"] for name, df in per_query.items()})
axes[1].imshow(rr.values, cmap="Greens", vmin=0, vmax=1, aspect="auto")
for i in range(rr.shape[0]):
    for j in range(rr.shape[1]):
        axes[1].text(j, i, f"{rr.values[i, j]:.2f}", ha="center", va="center", fontsize=8,
                     color="white" if rr.values[i, j] > 0.6 else "black")
axes[1].set_xticks(range(3), rr.columns)
axes[1].set_yticks(range(len(rr)), [f"{i} ({per_query['BM25'].loc[i, 'tags']})" for i in rr.index], fontsize=8)
axes[1].set_title("reciprocal rank per query (1 = right doc first)")
plt.tight_layout(); plt.show()

In [ ]:
by_tag = pd.DataFrame({name: df.groupby("tags")["RR"].mean() for name, df in per_query.items()})
print("MRR by question type:\n" + by_tag.round(3).to_string())

assert bm25_rank("What does error E-4012 mean?")[0] == "d10"                    # exact identifier: BM25 nails it
assert by_tag.loc["paraphrase", "dense"] > by_tag.loc["paraphrase", "BM25"]      # paraphrase: dense wins
assert summary.loc["BM25", "MRR"] < summary.loc["hybrid", "MRR"] < summary.loc["dense", "MRR"]   # here, plain RRF lands in between

**Reading it:**
- BM25 is perfect on identifiers and keyword questions and misses paraphrases: "money back" never matches "refunds", and "older copy of a document I overwrote" never matches "previous versions of a file".
- On this small corpus MiniLM is perfect, even on the near-duplicate error codes. On real corpora with thousands of codes, SKUs and names, dense retrieval loses exact identifiers far more often. That is the half of `ve13` this toy set cannot show.
- **Plain RRF hybrid lands in between.** Equal-weight fusion let a confidently wrong BM25 ranking push q10's right document out of the top 3. Hybrid is the usual production default because it is *robust across corpora*, not because it always beats its best component. **Measure it; don't assume it.** Weighting the fusion (Try it yourself 3), or adding a reranker, is how you get it back.
- The per-query heatmap is the view to read: an aggregate hides *which kind of question* broke. For RAG, report recall@k **at the k you actually pass to the model** (`ve29` trap).

## 4. Generation: answer the golden questions

A minimal RAG step: hybrid top-3 chunks, each tagged with its doc id, and a prompt that says to use only the context and to say "I don't know" otherwise. Offline, the stand-in answers extractively: it quotes the best-matching context sentence with its `[doc id]`. That happens to be a perfectly grounded strategy, so the offline answers are faithful by construction. A real model paraphrases and can drift, which is what the metrics below exist to catch.

In [ ]:
K = 3
PROMPT = ("Answer the question using only the context below. Cite the doc id in brackets. "
          "If the context does not contain the answer, say you don't know.\n\nContext:\n{context}\n\nQuestion: {question}")

def retrieve(q, k=K):
    return hybrid_rank(q)[:k]

def rag_answer(q, k=K):
    ids = retrieve(q, k)
    context = "\n".join(f"[{d}] {KB[d]}" for d in ids)
    resp = client.chat.completions.create(model=MODEL, temperature=0,
                                          messages=[{"role": "user", "content": PROMPT.format(context=context, question=q)}])
    return resp.choices[0].message.content, ids

outputs = {g["id"]: rag_answer(g["q"]) for g in GOLDEN}
for g in GOLDEN[:4] + GOLDEN[-1:]:
    print(f"{g['id']} {g['q']}\n    -> {outputs[g['id']][0][:110]}   (retrieved {outputs[g['id']][1]})")

## 5. Faithfulness (groundedness): are the answer's claims in the context?

RAGAS-style faithfulness (`ev07`, `ev14`) works in three steps: split the answer into atomic claims, check each claim against the retrieved context, and report supported claims / all claims. It needs **no reference answer**, so it can run on live traffic.

Our simple, transparent checker scores each claim three ways:
- **Lexical support:** the share of the claim's content words that appear in the best-matching context sentence.
- **Embedding support:** the maximum MiniLM cosine between the claim and any context sentence.
- **Numbers check:** every number in the claim must appear somewhere in the context. This is the mechanical check from `ev14`: free, deterministic, and it catches a surprising share of hallucinations.

A claim is supported if its numbers check out *and* either lexical support ≥ 0.6 or embedding support ≥ 0.7. These thresholds were set by eye on this toy set. In a real system, calibrate them on a labelled sample, or replace this checker with an NLI model or an LLM judge (next cell but one).

In [ ]:
WORD = re.compile(r"[a-z0-9$]+")
def content_words(text):
    return {w for w in WORD.findall(text.lower()) if w not in STOP}

def split_claims(answer: str) -> list[str]:
    """Crude claim splitter: strip [doc] tags, split into sentences, then on ';' and ', and'."""
    text = re.sub(r"\[d\d+\]", "", answer)
    parts = re.split(r"(?<=[.!?])\s+|;\s*|,\s+and\s+|\.\s+Also,?\s+", text)
    return [p.strip(" .") for p in parts if len(content_words(p)) >= 2]

def sentences(text):
    return [s for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]

def claim_support(claim: str, context: str) -> dict:
    sents = sentences(context)
    cw = content_words(claim)
    lexical = max(len(cw & content_words(s)) / len(cw) for s in sents)
    emb = float(max(embed_cached(sents) @ embed_one(claim)))
    nums_ok = all(n in context for n in re.findall(r"\d+(?:\.\d+)?", claim))
    return {"claim": claim, "lexical": lexical, "embedding": emb, "numbers_ok": nums_ok,
            "supported": nums_ok and (lexical >= 0.6 or emb >= 0.7)}

def faithfulness(answer, context):
    checks = [claim_support(c, context) for c in split_claims(answer)]
    return (np.mean([c["supported"] for c in checks]) if checks else float("nan")), checks

ctx_q01 = " ".join(KB[d] for d in retrieve(GOLDEN[0]["q"]))
demo_answers = {
    "faithful": "Refunds go to your original payment method within 5 business days after we receive the returned item.",
    "half invented": "Refunds go to your original payment method within 5 business days. Also, you receive a 10% discount code for your next order.",
    "invented": "Refunds are paid in cash at any of our stores within 24 hours.",
}
faith_scores = {}
for name, ans in demo_answers.items():
    faith_scores[name], checks = faithfulness(ans, ctx_q01)
    print(f"{name:14} faithfulness = {faith_scores[name]:.2f}")
    for c in checks:
        print(f"    {'OK ' if c['supported'] else 'BAD'} lex={c['lexical']:.2f} emb={c['embedding']:.2f} nums_ok={c['numbers_ok']!s:5}  {c['claim']!r}")
assert faith_scores["faithful"] == 1.0 and faith_scores["half invented"] == 0.5 and faith_scores["invented"] == 0.0

### The LLM-judge version

Production faithfulness checkers (RAGAS, DeepEval, TruLens groundedness) ask an LLM to do both steps: decompose the answer into claims, then verdict each claim against the context. With structured output that is one call per answer.

In [ ]:
class ClaimCheck(BaseModel):
    claim: str
    supported: bool

class FaithfulnessReport(BaseModel):
    claims: list[ClaimCheck]

JUDGE_SYSTEM = ("Split the ANSWER into short atomic factual claims. For each claim, decide whether the CONTEXT "
                "supports it. A claim with any detail missing from the context is not supported.")

def llm_faithfulness(answer, context):
    resp = client.chat.completions.parse(model=MODEL, temperature=0, response_format=FaithfulnessReport, messages=[
        {"role": "system", "content": JUDGE_SYSTEM}, {"role": "user", "content": f"CONTEXT:\n{context}\n\nANSWER:\n{answer}"}])
    report = resp.choices[0].message.parsed
    return np.mean([c.supported for c in report.claims]) if report.claims else float("nan"), report

for name, ans in demo_answers.items():
    score, report = llm_faithfulness(ans, ctx_q01)
    print(f"{name:14} LLM faithfulness = {score:.2f}  ({len(report.claims)} claims)  heuristic = {faith_scores[name]:.2f}")
if PROVIDER == "offline":
    print("\n(offline stand-in: it returns one canned claim and does not really check it. With gpt-4.1-mini, expect it to "
          "agree with the heuristic on these clear cases and do better on paraphrased claims.)")

## 6. Answer relevancy: does the answer address the question?

An answer can be perfectly faithful to the context and still dodge the question. The simplest reference-free check is the cosine between the question and the answer. **RAGAS** does it the other way round: an LLM writes the questions the answer *would* answer, and those are compared with the real question. That catches answers that are fluent, grounded and about something else. Both versions are below. The LLM version makes one call.

In [ ]:
q = GOLDEN[0]["q"]
candidates = {"answers it": "Refunds reach your original payment method within 5 business days of the return arriving.",
              "grounded but evasive": "Items must be returned within 30 days of delivery.",
              "off topic": "Live chat support is open Monday to Friday."}
rel_scores = {k: float(embed_one(q) @ embed_one(a)) for k, a in candidates.items()}
for k, v in rel_scores.items():
    print(f"answer relevancy (cosine) {k:22} = {v:.3f}")
assert rel_scores["answers it"] > rel_scores["grounded but evasive"] > rel_scores["off topic"]

gen = client.chat.completions.create(model=MODEL, temperature=0, messages=[{"role": "user", "content":
      f"Write the one question that this text answers. Reply with the question only.\n\nText: {candidates['grounded but evasive']}"}])
generated_q = gen.choices[0].message.content
print(f"\nRAGAS-style: generated question = {generated_q[:90]!r}\n  cosine to the real question = {float(embed_one(q) @ embed_one(generated_q)):.3f}")

## 7. Context precision and context recall (RAGAS-style)

These two grade the **retrieved context** rather than the answer (`rg30`):

- **Context precision@K** (rank-aware): Σ_k precision@k · v_k / (number of relevant chunks in the top K), where v_k = 1 if chunk k is relevant. It is average precision over the retrieved list, and it rewards putting relevant chunks first. Hand example: relevance [1, 0, 1] gives (1/1 + 2/3)/2 = **0.833**.
- **Context recall:** the share of the **reference answer's** claims that the retrieved context supports. It is the only one of the four RAGAS metrics that needs ground truth, which is why it is the one teams skip (`ev07`).

RAGAS computes both with an LLM deciding relevance and support. We use the golden labels and the claim checker from section 5.

In [ ]:
def context_precision(ranked_ids, rel, k):
    v = [1.0 if d in rel else 0.0 for d in ranked_ids[:k]]
    hits = sum(v)
    return sum((sum(v[:i + 1]) / (i + 1)) * v[i] for i in range(len(v))) / hits if hits else 0.0

def context_recall(reference, ranked_ids, k):
    context = " ".join(KB[d] for d in ranked_ids[:k])
    checks = [claim_support(c, context) for c in split_claims(reference)]
    return float(np.mean([c["supported"] for c in checks]))

assert abs(context_precision(["a", "x", "b"], {"a", "b"}, 3) - (1 + 2 / 3) / 2) < 1e-12
assert context_precision(["x", "a", "b"], {"a", "b"}, 3) < context_precision(["a", "b", "x"], {"a", "b"}, 3)   # rank-aware

ks = range(1, 7)
curves = {"context recall": [], "context precision": [], "precision@k": []}
for k in ks:
    rows = [(context_recall(g["ref"], hybrid_rank(g["q"]), k), context_precision(hybrid_rank(g["q"]), g["rel"], k),
             precision_at_k(hybrid_rank(g["q"]), g["rel"], k)) for g in ANSWERABLE]
    for name, col in zip(curves, zip(*rows)):
        curves[name].append(float(np.mean(col)))
print({k_: [round(x, 2) for x in v] for k_, v in curves.items()})
assert all(b >= a for a, b in zip(curves["context recall"], curves["context recall"][1:]))   # more context never loses claims
assert curves["precision@k"][-1] < curves["precision@k"][0]                                   # ...but adds noise

fig, ax = plt.subplots(figsize=(8, 3.8))
for name, vals in curves.items():
    ax.plot(list(ks), vals, "o-", label=name)
ax.set_xlabel("k (chunks in the prompt)"); ax.set_ylabel("mean over golden set"); ax.set_ylim(0, 1.05)
ax.set_title("Recall and precision trade off through k: report both, plus the k"); ax.legend(); plt.show()

## 8. The full scorecard for the pipeline

One row per golden question, both halves side by side. This is the table you would diff between two versions of the system. Predict before running: which question will fail, and in which half?

In [ ]:
rows = []
for g in GOLDEN:
    answer, ids = outputs[g["id"]]
    context = " ".join(KB[d] for d in ids)
    refused = bool(re.search(r"don.?t know", answer, re.I))
    faith, _ = faithfulness(answer, context) if not refused else (float("nan"), [])
    rows.append({"id": g["id"], "tags": ",".join(g["tags"]),
                 "hit@3": hit_rate_at_k(ids, g["rel"], 3) if g["rel"] else float("nan"),
                 "ctx precision@3": context_precision(ids, g["rel"], 3) if g["rel"] else float("nan"),
                 "ctx recall@3": context_recall(g["ref"], ids, 3) if g["rel"] else float("nan"),
                 "faithfulness": faith, "relevancy": float(embed_one(g["q"]) @ embed_one(answer)),
                 "refused": refused, "should refuse": not g["rel"]})
card = pd.DataFrame(rows).set_index("id")
print(card.round(2).to_string())
print("\nmeans:", card[["hit@3", "ctx precision@3", "ctx recall@3", "faithfulness", "relevancy"]].mean().round(3).to_dict())
wrong_refusals = card.index[card.refused & ~card["should refuse"]].tolist()
print("refused although answerable:", wrong_refusals, "| out-of-scope refused:", bool(card.loc["q14", "refused"]))
assert card.loc["q14", "refused"]                                        # the out-of-scope question gets "I don't know"
assert all(card.loc[q_, "hit@3"] == 0 for q_ in wrong_refusals)           # every wrong refusal is a retrieval miss

**Diagnostic reading (`ev07`):**

| Pattern | Diagnosis | Fix |
|---|---|---|
| low context recall | the retriever missed needed facts | chunking, hybrid search, query rewriting, bigger k + reranker |
| high recall, low context precision | right chunks buried in noise | reranker, smaller k |
| high recall, low faithfulness | the model ignores or embellishes the context | prompt ("only the context"), citations, a stronger model |
| high faithfulness, low relevancy | grounded but answering a different question | prompt, query understanding |
| answers the out-of-scope question | no refusal path | an "I don't know" rule, empty-retrieval handling, out-of-scope eval cases |

Our scorecard's one failure, **q10**, is the first row of that table. Context recall is 0 and hit@3 is 0, and the generator did the right thing *given what it saw*: it said it didn't know. Blaming the model would be wrong. The fix is retrieval. That is `rg11`'s trap in one row: check the retrieval columns before you touch the prompt.

Accuracy and faithfulness come apart (`ev30`). A **true but unfaithful** answer came from the model's memory and will be wrong the day your policy differs from the internet. A **faithful but inaccurate** answer means the source document is wrong.

## 9. Building and maintaining the golden set

**How to build it (`rg12`, `ev05`):**
1. Source 50–100 **real** questions: support tickets, search logs, chat transcripts. Invented questions test an imaginary product.
2. Label the answering chunk(s) for each. That gives recall@k and MRR forever, with no LLM.
3. Write reference answers for at least a subset, for correctness and context recall.
4. Stratify deliberately: paraphrase, identifier, multi-hop, numbers, **out-of-scope**, one per language.
5. You can bootstrap with an LLM (generate questions from chunks), but **verify by hand**. Unverified synthetic sets measure your generator.

**How to keep it useful (`ev15`):** freeze and version it, and record the version with every score. Add every production failure as a case *before* fixing it. Keep a held-out slice no prompt tuning ever sees. Audit periodically for stale answers.

In [ ]:
def golden_version(golden) -> str:
    """Content hash of the golden set: store it next to every score so results stay interpretable."""
    return hashlib.sha256(json.dumps(golden, sort_keys=True, default=sorted).encode()).hexdigest()[:12]

v1 = golden_version(GOLDEN)
new_case = {"id": "q15", "q": "Who pays customs fees on international orders?", "rel": {"d06": 2},
            "ref": "The customer pays customs duties and import taxes on delivery.", "tags": ["from-production-bug"]}
GOLDEN_V2 = GOLDEN + [new_case]
v2 = golden_version(GOLDEN_V2)
print(f"golden set v1 = {v1} ({len(GOLDEN)} cases)  ->  v2 = {v2} ({len(GOLDEN_V2)} cases)")
assert v1 != v2 and golden_version(GOLDEN) == v1          # any edit changes the version; same content = same version

counts = Counter(t for g in GOLDEN_V2 for t in g["tags"])
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(counts.keys(), counts.values(), color="#3b5bdb")
ax.set_ylabel("cases"); ax.set_title("Stratification check: does every failure class have cases?")
plt.xticks(rotation=20); plt.tight_layout(); plt.show()

## 10. Tools (not installed here; know what each is for)

| Tool | What it is | Typical use | Notes |
|---|---|---|---|
| **RAGAS** | Python library of RAG metrics (faithfulness, answer relevancy, context precision/recall, and more) | offline scoring of a golden set in CI | LLM-judged, so it carries judge noise and cost; pin the judge model |
| **TruLens** | feedback functions plus tracing; coined the "RAG Triad" | groundedness / context relevance / answer relevance on traces | good for monitoring as well as offline runs |
| **DeepEval** | pytest-style LLM eval framework | assertions like faithfulness ≥ 0.8 in unit tests | fits "evals as tests" in CI |
| **Arize Phoenix** | open-source tracing and eval UI (OpenTelemetry-based) | inspect retrievals per request, run evals on traces | self-hostable |
| **LangSmith** | LangChain's hosted tracing, datasets and eval runs | versioned datasets, run comparisons, annotation queues | tightest with LangChain/LangGraph apps |

Whatever the tool, the pieces are the ones you just built: a versioned case set, retrieval metrics computed without an LLM, a faithfulness check, and a per-case diff between runs.

## Try it yourself

**1. Hit rate vs recall.** For q12 (multi-hop: SSO **and** the Pro plan) and q10 (versions, with trash as related), compare hit@3 and recall@3 for each retriever. Why do they differ here but not on the other questions?

In [ ]:
# Solution — try it yourself first, then run this
for gid in ["q12", "q10"]:
    g_ = next(g for g in GOLDEN if g["id"] == gid)
    for name, f in RETRIEVERS.items():
        r = f(g_["q"])
        print(f"{gid} {name:7} top-3 {r[:3]}  hit@3={hit_rate_at_k(r, g_['rel'], 3):.0f}  recall@3={recall_at_k(r, g_['rel'], 3):.2f}")
# Hit rate is satisfied by one relevant doc; recall needs both. With one relevant doc per query they coincide.

**2. Mechanical hallucination checks.** Write `citation_check(answer, retrieved_ids)`: every `[dNN]` cited in the answer must be one of the retrieved ids. Run it on the pipeline outputs and on a fake answer that cites `[d99]`.

In [ ]:
# Solution — try it yourself first, then run this
def citation_check(answer, retrieved_ids):
    cited = re.findall(r"\[(d\d+)\]", answer)
    return all(c in retrieved_ids for c in cited), cited

print("pipeline answers pass:", all(citation_check(a, ids)[0] for a, ids in outputs.values()))
print("fake answer:", citation_check("Refunds take 5 days [d99].", ["d01", "d08", "d02"]))

**3. Tune the hybrid.** Fusion has two knobs: the weight on each retriever and the RRF constant `k_rrf`. A large k flattens rank differences, so two retrievers agreeing on a mediocre doc can beat one retriever's confident first place. Grid-search `w_dense` ∈ {1, 2, 4} × `k_rrf` ∈ {60, 10, 1}. Report MRR over the golden set and where q10's right document (d19) lands. Which setting would you ship, and what would you check it on before shipping?

In [ ]:
# Solution — try it yourself first, then run this
g10 = next(g for g in GOLDEN if g["id"] == "q10")
for k_rrf in [60, 10, 1]:
    for w in [1.0, 2.0, 4.0]:
        f = lambda q_: hybrid_rank(q_, k_rrf=k_rrf, w_dense=w)
        mrr = np.mean([reciprocal_rank(f(g["q"]), g["rel"]) for g in ANSWERABLE])
        print(f"k_rrf={k_rrf:2d}  w_dense={w:.0f}  MRR={mrr:.3f}  q10: d19 at rank {f(g10['q']).index('d19') + 1}")
# Fusion settings are hyperparameters: tune them on the golden set, then confirm on a held-out slice -
# with 13 questions, one flipped query moves MRR by ~0.04, so small differences here are noise (notebook 05).

## Say it in an interview

**30-second answer (`rg11`):** "I split it into two halves. Retrieval I score without any LLM, on a labelled golden set: hit rate or recall at the k I actually pass to the model, MRR, nDCG, and context precision. Generation I score for faithfulness, meaning each claim in the answer is supported by the retrieved context. That needs no reference, so it can run on production traffic. I add answer relevancy, and correctness where I have references. The golden set is 50–100 real questions, stratified to include identifiers, multi-hop and out-of-scope cases, frozen and versioned, and it grows from production failures."

**Definitions to have ready:** hit@k (any relevant in top k) vs recall@k (share of relevant retrieved). MRR = mean 1/rank of the first hit. nDCG = DCG / ideal DCG with log2 discount. Context precision = rank-aware precision of the retrieved chunks. Context recall = share of the reference answer's claims supported by the context. Faithfulness = supported claims / claims.

**Follow-ups:**
- *Which metric first?* recall@k, because the generator only sees k.
- *Recall is high but answers are wrong?* Check faithfulness and context precision: noise or embellishment.
- *No labels?* Start with 50 hand-labelled questions (a day of work). Synthetic questions from chunks work as a bootstrap only if you verify them.
- *RAGAS in production?* Faithfulness yes (reference-free). Context recall only offline (needs references). All of them are LLM-judged, so treat them as directional and keep deterministic checks underneath.

**Traps:**
- Reporting nDCG without recall@k at your real k.
- Blaming the model before checking whether the chunk was retrieved (`rg11` trap: index → top 50 → prompt → used).
- A golden set with no out-of-scope questions (`rg12`).
- Calling true-but-unfaithful answers a success (`ev30`).

## Next

- [04 · Agent evaluation](04_agent_evaluation.ipynb)
- [05 · A/B testing and statistics for evals](05_ab_testing_and_statistics_for_evals.ipynb): is hybrid's gain significant on 12 questions? (No.)
- [02 · LLM-as-judge and its biases](02_llm_as_judge_and_its_biases.ipynb): the judge behind LLM-based faithfulness.
- [BM25 from scratch](../09_rag/02_bm25_from_scratch.ipynb) · [Hybrid search and RRF](../09_rag/04_hybrid_search_and_rrf.ipynb) · [Reranking](../09_rag/05_reranking.ipynb)
- [RAG skeleton (interview round)](../17_interview_coding_rounds/01_rag_skeleton.ipynb)
- Theory: [genai_flow course](../../genai_flow/index.html) · [AI system design concepts](../../ai_system_design_concepts/index.html) · [interview bank](../../ai_interview_prep/index.html)